# The AI Telco Troubleshooting Challenge

Goal: Enhance the accuracy of Qwen2.5-1.5B when answering telco troubleshooting questions in telelogs data.

## I/ Set-up

### 1. Load libraries

In [1]:
import pandas as pd
import numpy as np
import re
import random
from io import StringIO

### 2.Load data

In [2]:
#load data
df = pd.read_csv("/home/onyxia/work/Telco-challenge/data/raw/train.csv")
print(df.head())
print(f"Number of rows {df.shape[0]}")

              ID                                           question answer
0  ID_1P7PJMPV0R  Analyze the 5G wireless network drive-test use...     C2
1  ID_8B1D1TUTFA  Analyze the 5G wireless network drive-test use...     C1
2  ID_IGGXMA9GZH  Analyze the 5G wireless network drive-test use...     C2
3  ID_D6C9N2X295  Analyze the 5G wireless network drive-test use...     C2
4  ID_8JC15PNP3Q  Analyze the 5G wireless network drive-test use...     C5
Number of rows 2400


In [3]:
print(df["question"][0])

Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values represent the ac

**Game plan**
We have time series in drive test data: multiple timestamps by telelog Id and engineering data linked to PCI that we also find in the drive test data so:

1) Parse "question" columns into two categories of features: drive test and engineering. Then merge to have a final dataframe which is a structured like a time series.
2) Train time series/sequential ML models on the features to predict root cause
3) If possible fine-tune an LLM to find the root cause directly from the "question" columns
4) Compare both approaches

### 3.Preprocess data

In [11]:
import pandas as pd
import numpy as np
from io import StringIO


class Preprocessor():

    def __init__(self, question):

        self.question = question

    def parse_question(self):
        """
        Divide question into:
        - drive test dataframe
        - engineering dataframe
        """

        # Split sections
        parts = self.question.split(
            "Engeneering parameters data as follows："
        )

        drive_part = parts[0]
        eng_part = parts[1]

        # -------------------------
        # DRIVE TEST TABLE
        # -------------------------

        drive_lines = []

        started = False

        for line in drive_part.split("\n"):

            if "Timestamp|" in line:
                started = True

            if started and "|" in line:
                drive_lines.append(line)

        drive_text = "\n".join(drive_lines)

        drive_df = pd.read_csv(
            StringIO(drive_text),
            sep="|"
        )
        drive_df.columns = drive_df.columns.str.strip()
        # -------------------------
        # ENGINEERING TABLE
        # -------------------------

        eng_lines = []

        started = False

        for line in eng_part.split("\n"):

            if "gNodeB ID|" in line:
                started = True

            if started and "|" in line:
                eng_lines.append(line)

        eng_text = "\n".join(eng_lines)

        eng_df = pd.read_csv(
            StringIO(eng_text),
            sep="|"
        )
        eng_df.columns = eng_df.columns.str.strip()
        return drive_df, eng_df

    def merge_features(self, drive_df, eng_df):
        """
        Merge categories of features into one dataframe
        """

        merged = drive_df.merge(
            eng_df,
            left_on="5G KPI PCell RF Serving PCI",
            right_on="PCI",
            how="left"
        )

        return merged

    def build_sequence(self):

        """
        Transform the dataframes into one row by telelog id
        """

        drive_df, eng_df = self.parse_question()

        merged = self.merge_features(
            drive_df,
            eng_df
        )
        merged.columns = merged.columns.str.strip()
        # replace "-" with NaN
        merged = merged.replace("-", np.nan)

        # convert numeric
        for col in merged.columns:

            merged[col] = pd.to_numeric(
                merged[col],
                errors="coerce"
            )

        # fill missing values
        merged = merged.fillna(0)

        sequence = merged[
            merged.columns
        ].values

        return sequence

In [ ]:
#Let's process the data
processed_data = []

for idx, row in df.iterrows():

    try:

        processor = Preprocessor(row["question"])

        sequence = processor.build_sequence()

        processed_data.append({
            "ID": row["ID"],
            "sequence": sequence,
            "label": row["answer"]
        })

    except Exception as e:

        print(f"Error on row {idx}: {e}")
processed_data[0]

In [ ]:
# #Function to clean questions
# def clean_question(question):
#     lines=question.split("\n")
#     content=[]
#     for line in lines:
#         if "|" in line:
#             content.append(line.split("|"))
#     drive_test_data=[{"Observation": i+1,**dict(zip(content[0],row))} for i,row in enumerate(content[1:])]
#     engineering_params=[{"Observation": i+1,**dict(zip(content[11],row))} for i,row in enumerate(content[12:])]
#     #clean question
#     ##recompute begining of the prompt
#     q=""
#     for l in [l+"\n" for l in lines[:19]]:
#         q=q+l
#     ##assemble
#     cleaned_question= f" Question: {q} \nDrive test data: {drive_test_data} \nEngineering parameters {engineering_params} "
#     return cleaned_question
# #Apply to dataset
# df["cleaned_question"]=df["question"].apply(lambda x: clean_question(x))
# print(df["cleaned_question"][0])

In [ ]:
def compute_features(question):
    def extract_features(question):
        lines = question.split("\n")
        content = []
        for line in lines:
            if "|" in line:
                content.append(line.split("|"))
    
        headers = content[0]
        rows = content[1:11]  # only drive test rows
    
        drive_test = [dict(zip(headers, row)) for row in rows]
    
        return drive_test
    data = extract_features(question)
    
    # convert to numeric safely
    speeds = [float(d["GPS Speed (km/h)"]) for d in data]
    sinr = [float(d["5G KPI PCell RF Serving SS-SINR [dB]"]) for d in data]
    throughput = [float(d["5G KPI PCell Layer2 MAC DL Throughput [Mbps]"]) for d in data]
    rb = [float(d["5G KPI PCell Layer1 DL RB Num (Including 0)"]) for d in data]
    
    features = {
        "avg_speed": np.mean(speeds),
        "max_speed": np.max(speeds),
        "avg_sinr": np.mean(sinr),
        "min_sinr": np.min(sinr),
        "avg_throughput": np.mean(throughput),
        "min_throughput": np.min(throughput),
        "avg_rb": np.mean(rb),
        "min_rb": np.min(rb),
        "throughput_below_600_ratio": sum(t < 600 for t in throughput) / len(throughput),
    }
    
    return features
    def compute_engineering_features(question):
        params = extract_engineering_params(question)
        downtilts = []
        pcis = []
        heights = []
    
        for p in params:
            try:
                downtilt = float(p["Mechanical Downtilt"]) + float(p["Digital Tilt"])
                downtilts.append(downtilt)
            
                pcis.append(int(p["PCI"]))
                heights.append(float(p["Height"]))
            except:
                continue
    
        features = {
            "avg_downtilt": sum(downtilts)/len(downtilts) if downtilts else 0,
            "max_downtilt": max(downtilts) if downtilts else 0,
            "min_downtilt": min(downtilts) if downtilts else 0,
            "num_cells": len(params),
            "unique_pci": len(set(pcis)),
            "pci_mod30_conflict": len(set([p % 30 for p in pcis])) < len(pcis),
            "avg_height": sum(heights)/len(heights) if heights else 0
        }
    
    return features
    
features_df = df["question"].apply(compute_features).apply(pd.Series)

In [ ]:
features_df.columns

Index(['avg_speed', 'max_speed', 'avg_sinr', 'min_sinr', 'avg_throughput',
       'min_throughput', 'avg_rb', 'min_rb', 'throughput_below_600_ratio'],
      dtype='str')

In [ ]:
a

NameError: name 'a' is not defined

## II/ Untrained model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from dotenv import load_dotenv
import os
from datasets import Dataset

/home/onyxia/work/Telco-challenge/telco_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#empty cache
torch.cuda.empty_cache()
#set token 
load_dotenv("telco.env")
token = os.getenv("HF_TOKEN")

In [ ]:

# load the tokenizer and the model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="sequential",
    offload_folder="offload"
)
model.eval()

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 426.04it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
def batch_predict(batch):
    questions = batch["cleaned_question"]

    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": q}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        for q in questions
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32
        )

    predictions = []

    for i in range(len(prompts)):
        decoded = tokenizer.decode(
            outputs[i][inputs["input_ids"][i].shape[0]:],
            skip_special_tokens=True
        )

        match = re.search(r"\b(C\d)\b", decoded)
        predictions.append(match.group(1) if match else "UNKNOWN")

    return {"root_cause": predictions}
      

In [ ]:
# Select rows to test model
df_test=df.iloc[[random.randint(0, 2400-1) for _ in range(10)]]
ds_test = Dataset.from_pandas(df_test)

In [ ]:
#predictions
ds_test = ds_test.map(
    batch_predict,
    batched=True,
    batch_size=8
)
#save dataset
ds_test.save_to_disk("checkpoint_ds")
df_test = ds_test.to_pandas()


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
git add .

In [ ]:
git commit -m "Saved predictions to disk"

In [ ]:
git push

## III/ Fine-tune model

In [ ]:
import time
from sagemaker.huggingface import HuggingFace


job_name = f"mixtral-8x7b-qlora-{time.strftime('%Y-%m-%d-%H-%M-%S', time.localtime())}"

hyperparameters = {
    "model_id": model_id,
    "dataset_path": "/opt/ml/input/data/training",
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "lr": 2e-4,
    "merge_weights": True,
}

huggingface_estimator = HuggingFace(
    entry_point = "run_clm.py",
    source_dir= "scripts",
    instance_type = "ml.g5.24xlarge",
    instance_count = 1,
    base_job_name = job_name,
    role = role,
    volume_size = 300,
    transformers_version= "4.28",
    pytorch_version= "2.0",
    py_version= "py310",
    hyperparameters = hyperparameters,
    environment = {
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache"

_IncompleteInputError: incomplete input (2506589804.py, line 29)

In [ ]:
data = {"training": training_input_path}
huggingface_estimator.fit(data,wait = True)